In [1]:
# ==============================================================================
# ConMamba-ASR — Q1 NEW EXPERIMENTS SUITE v2 (Bug-Free & 3x Faster on Dual-T4)
# ==============================================================================

import os, sys, re, glob, json, time, math, queue, shutil, hashlib, platform, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==============================================================================
# 0. PATCH TORCHAUDIO IN MAIN PROCESS BEFORE SPEECHBRAIN IMPORT
# ==============================================================================
import torch
import torchaudio
if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]
if not hasattr(torchaudio, "set_audio_backend"):
    torchaudio.set_audio_backend = lambda *args, **kwargs: None

# ==============================================================================
# 1. EXPERIMENT CONFIGURATION
# ==============================================================================
EXISTING_CKPTS_DIR = None

RUN_WHISPER_LORA      = True   # ~20 min on 1x T4 (Pure PyTorch FP16)
RUN_NEW_ABLATIONS     = True   # Runs K=7 and Pure BiMamba (No-Conv) in parallel
RUN_LATENCY_BENCHMARK = True   # ~30 sec (5s -> 60s Latency & Peak VRAM)
RUN_CKPT_INFERENCE    = True   # Runs Table IV + Noise Stress Test if ckpts are found

ABLATION_EPOCHS       = 80     
TRAIN_BATCH_SIZE      = 64     
TRAIN_LR              = 0.0010 
WARMUP_STEPS          = 10000  
VALID_SEARCH_INTERVAL = 80     
TEST_BEAM_SIZE        = 5
TIME_LIMIT_H          = 11.2

WHISPER_MODEL  = "openai/whisper-small"
WHISPER_EPOCHS = 3             
WHISPER_LR     = 1e-3
WHISPER_BS     = 16            

LONG_STRATEGIES = ["unchunked", "fixed", "chunked", "overlap_energy"]
SNR_LEVELS_DB   = [20, 10, 5, 0]

COMPLETED_MAIN = [
    {"model": "Whisper-Small (Zero-Shot)", "params_m": 241.7, "test_wer": 71.63, "test_cer": 33.64, "test_long_wer": 53.62, "test_long_cer": 21.16, "rtf_long": 0.0966},
    {"model": "ConMambaMamba-Small",       "params_m": 12.2,  "test_wer": 17.97, "test_cer": 6.41,  "test_long_wer": 19.82, "test_long_cer": 7.44,  "rtf_long": 0.0512},
    {"model": "Conformer-Small",           "params_m": 12.5,  "test_wer": 14.49, "test_cer": 4.77,  "test_long_wer": 15.38, "test_long_cer": 4.82,  "rtf_long": 0.0624},
    {"model": "ConMamba-Small (Ours)",     "params_m": 12.1,  "test_wer": 13.85, "test_cer": 4.44,  "test_long_wer": 14.65, "test_long_cer": 4.57,  "rtf_long": 0.0438},
]

COMPLETED_ABLATIONS = [
    {"Variant": "ConMamba-Small (Full: Bi-SSM + K=31 + CTC a=0.3)", "Params (M)": 12.1, "Test WER (%)": 13.85, "Test CER (%)": 4.44, "Long WER (%)": 14.65},
    {"Variant": "w/o Bidirectional (Unidirectional SSM)",           "Params (M)": 10.6, "Test WER (%)": 14.97, "Test CER (%)": 4.81, "Long WER (%)": 17.22},
    {"Variant": "w/o CTC Loss (Pure S2S, a=0.0)",                   "Params (M)": 12.1, "Test WER (%)": 251.0, "Test CER (%)": 264.0,"Long WER (%)": 282.0},
]

WORK = "/kaggle/working"
REPO = f"{WORK}/Mamba-ASR"
RESULTS_ROOT = f"{WORK}/q1_new_results_v2"
os.makedirs(RESULTS_ROOT, exist_ok=True)

# ==============================================================================
# 2. ENVIRONMENT & DEPENDENCY SETUP
# ==============================================================================
import numpy as np
import pandas as pd

NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS >= 1, "No GPU detected! Enable Accelerator -> GPU T4 x2."
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | GPUs: {NUM_GPUS}x {torch.cuda.get_device_name(0)}")

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Remove incompatible pre-installed torchao
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])

pip_install([
    "speechbrain==1.0.3", "sentencepiece", "einops", "soundfile",
    "packaging", "ninja", "jiwer", "scipy",
])
pip_install(["causal-conv1d", "mamba-ssm", "--no-build-isolation"])
if RUN_WHISPER_LORA:
    pip_install(["transformers", "accelerate", "peft"])

if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "-q", "https://github.com/xi-j/Mamba-ASR.git", REPO])

avail_yamls = sorted([Path(p).stem for p in glob.glob(f"{REPO}/hparams/S2S/*.yaml")])
print(f"Available Mamba-ASR S2S templates: {avail_yamls}")

# ==============================================================================
# 3. WRITE ejust_tools.py (COMPATIBILITY SHIM + STRICT YAML INJECTOR)
# ==============================================================================
ejust_tools_code = r'''
from pathlib import Path
import re

SHIM = """# --- compatibility shim (EJUST) ---
try:
    from causal_conv1d.cpp_functions import (
        causal_conv1d_fwd_function as _cc_new_fwd,
        causal_conv1d_bwd_function as _cc_new_bwd,
    )
    def _cc_fwd(x, weight, bias, seq_idx, silu):
        return _cc_new_fwd(x, weight, bias, seq_idx, None, None, silu)
    def _cc_bwd(x, weight, bias, dout, seq_idx, dx, silu):
        dx, dweight, dbias, _ = _cc_new_bwd(x, weight, bias, dout, seq_idx, None, None, dx, False, silu)
        return dx, dweight, dbias
except ImportError:
    import causal_conv1d_cuda
    def _cc_fwd(x, weight, bias, seq_idx, silu):
        try:
            return causal_conv1d_cuda.causal_conv1d_fwd(x, weight, bias, seq_idx, None, None, silu)
        except TypeError:
            return causal_conv1d_cuda.causal_conv1d_fwd(x, weight, bias, seq_idx, silu)
    def _cc_bwd(x, weight, bias, dout, seq_idx, dx, silu):
        try:
            r = causal_conv1d_cuda.causal_conv1d_bwd(x, weight, bias, dout, seq_idx, None, None, dx, False, silu)
            return r[0], r[1], r[2]
        except TypeError:
            return causal_conv1d_cuda.causal_conv1d_bwd(x, weight, bias, dout, seq_idx, dx, silu)
"""

LN_OLD = "    from mamba_ssm.ops.triton.layernorm import RMSNorm, layer_norm_fn, rms_norm_fn\n"
LN_NEW = (
    "    try:\n"
    "        from mamba_ssm.ops.triton.layernorm import RMSNorm, layer_norm_fn, rms_norm_fn\n"
    "    except ImportError:\n"
    "        from mamba_ssm.ops.triton.layer_norm import RMSNorm, layer_norm_fn, rms_norm_fn\n"
)

def patch_repo(repo):
    repo = Path(repo)
    p = repo / "modules/mamba/selective_scan_interface.py"
    if p.exists():
        s = p.read_text()
        if "_cc_fwd" not in s:
            s = re.sub(r"causal_conv1d_cuda\.causal_conv1d_fwd\((x, conv1d_weight)", r"_cc_fwd(\1", s)
            s = s.replace("causal_conv1d_cuda.causal_conv1d_bwd(\n", "_cc_bwd(\n")
            s = s.replace("import causal_conv1d_cuda\n", SHIM, 1)
            p.write_text(s)

    for f in ["modules/mamba/bimamba.py", "modules/mamba/mamba_blocks.py"]:
        p = repo / f
        if p.exists():
            s = p.read_text()
            if "except ImportError:" not in s and LN_OLD in s:
                p.write_text(s.replace(LN_OLD, LN_NEW, 1))

    p_tasr = repo / "modules/TransformerASR.py"
    if p_tasr.exists():
        s_tasr = p_tasr.read_text()
        old_tgt = "if self.num_decoder_layers > 0:\n            tgt = self.custom_tgt_module(tgt)"
        new_tgt = "if self.num_decoder_layers > 0 and tgt is not None:\n            tgt = self.custom_tgt_module(tgt)"
        if old_tgt in s_tasr:
            p_tasr.write_text(s_tasr.replace(old_tgt, new_tgt, 1))

    # Add no-conv bypass for Pure BiMamba ablation
    p_cm = repo / "modules/Conmamba.py"
    if p_cm.exists():
        s_cm = p_cm.read_text()
        if "self.no_conv" not in s_cm:
            s_cm = s_cm.replace(
                "        causal=False,\n        mamba_config=None\n    ):",
                "        causal=False,\n        mamba_config=None,\n        no_conv=False\n    ):\n        self.no_conv = no_conv"
            )
            s_cm = s_cm.replace(
                "        # convolution module\n        x = x + self.convolution_module(",
                "        # convolution module\n        if not getattr(self, 'no_conv', False):\n            x = x + self.convolution_module("
            )
            s_cm = s_cm.replace(
                "        causal=False,\n        mamba_config=None\n    ):\n        super().__init__()",
                "        causal=False,\n        mamba_config=None,\n        no_conv=False\n    ):\n        super().__init__()"
            )
            s_cm = s_cm.replace(
                "                    causal=causal,\n                    mamba_config=mamba_config,\n                )",
                "                    causal=causal,\n                    mamba_config=mamba_config,\n                    no_conv=no_conv,\n                )"
            )
            p_cm.write_text(s_cm)

    p_tr = repo / "modules/Transformer.py"
    if p_tr.exists():
        s_tr = p_tr.read_text()
        if "encoder_module == \"mamba\"" not in s_tr and "elif encoder_module == \"conmamba\":" in s_tr:
            mamba_branch = (
                "            elif encoder_module in [\"conmamba\", \"mamba\"]:\n"
                "                self.encoder = ConmambaEncoder(\n"
                "                    num_layers=num_encoder_layers,\n"
                "                    d_model=d_model,\n"
                "                    d_ffn=d_ffn,\n"
                "                    dropout=dropout,\n"
                "                    activation=branchformer_activation,\n"
                "                    kernel_size=kernel_size,\n"
                "                    bias=bias,\n"
                "                    causal=self.causal,\n"
                "                    mamba_config=mamba_config,\n"
                "                    no_conv=(encoder_module == \"mamba\")\n"
                "                )"
            )
            s_tr = re.sub(r"            elif encoder_module == \"conmamba\":\n                self\.encoder = ConmambaEncoder\([\s\S]+?mamba_config=mamba_config\n                \)", mamba_branch, s_tr, count=1)
            p_tr.write_text(s_tr)

    return True

def make_hparams(template, out_path, *, data_folder, csv_dir, bpe_model, output_folder,
                 epochs, warmup, valid_search_interval, test_beam_size=5,
                 batch_size=64, lr=0.0010, num_workers=4, grad_accumulation_factor=1,
                 long_strategies="chunked", snr_levels="", skip_train=False,
                 ablation_overrides=None):
    s = Path(template).read_text()
    extra_header = (
        f"data_folder: {data_folder}\n"
        f"bpe_model: {bpe_model}\n"
        f"long_strategies: {long_strategies}\n"
        f"snr_levels: \"{snr_levels}\""
    )
    rep = [
        (r"^data_folder: !PLACEHOLDER.*$", extra_header),
        (r"^skip_train: .*$", f"skip_train: {skip_train}"),
        (r"^output_folder: .*$", f"output_folder: {output_folder}"),
        (r"^train_csv: .*$", f"train_csv: {csv_dir}/train.csv"),
        (r"^valid_csv: .*$", f"valid_csv: {csv_dir}/dev.csv"),
        (r"^test_csv:\n(    - .*\n)+",
         f"test_csv:\n    - {csv_dir}/test.csv\n    - {csv_dir}/test_long.csv\n"),
        (r"^number_of_epochs: .*$", f"number_of_epochs: {epochs}"),
        (r"^grad_accumulation_factor: .*$", f"grad_accumulation_factor: {grad_accumulation_factor}"),
        (r"^num_workers: .*$", f"num_workers: {num_workers}"),
        (r"^precision: .*$", "precision: fp16"),
        (r"^lr_adam: .*$", f"lr_adam: {lr}"),
        (r"^batch_size: .*$", f"batch_size: {batch_size}"),
        (r"^dynamic_batching: .*$", "dynamic_batching: False"),
        (r"^valid_dataloader_opts:\n    batch_size: 1$", "valid_dataloader_opts:\n    batch_size: 64"),
        (r"^test_dataloader_opts:\n    batch_size: 1$", "test_dataloader_opts:\n    batch_size: 32"),
        (r"^output_neurons: .*$", "output_neurons: 500"),
        (r"^valid_search_interval: .*$", f"valid_search_interval: {valid_search_interval}"),
        (r"^test_beam_size: .*$", f"test_beam_size: {test_beam_size}"),
        (r"^    n_warmup_steps: .*$", f"    n_warmup_steps: {warmup}"),
    ]
    for pat, new in rep:
        s, n = re.subn(pat, new, s, count=1, flags=re.M)
        assert n == 1, f"{Path(template).name}: pattern not found: {pat}"

    s = re.sub(
        r"^(train_dataloader_opts:\n)",
        r"\1    pin_memory: True\n    persistent_workers: True\n",
        s, count=1, flags=re.M
    )

    if ablation_overrides:
        for k, v in ablation_overrides.items():
            val_str = str(v)
            pat = rf"^(\s*{re.escape(k)}:\s*).*$"
            s_new, n = re.subn(pat, rf"\g<1>{val_str}", s, flags=re.M)
            if n > 0:
                print(f"[ABLATION] {Path(out_path).name}: replaced {k} -> {val_str} ({n} match)")
                s = s_new
            else:
                s_inj, n_inj = re.subn(
                    r"^(Transformer:\s*!new:[^\n]+\n)",
                    rf"\1    {k}: {val_str}\n",
                    s, count=1, flags=re.M
                )
                assert n_inj == 1, f"Failed to inject ablation key '{k}' into Transformer block!"
                print(f"[ABLATION] {Path(out_path).name}: injected {k}: {val_str} into Transformer block")
                s = s_inj

    Path(out_path).write_text(s)
    return out_path
'''
Path(f"{WORK}/ejust_tools.py").write_text(ejust_tools_code)
sys.path.insert(0, WORK); sys.path.insert(0, REPO)
import ejust_tools
ejust_tools.patch_repo(REPO)
print("Patched Mamba-ASR successfully.")

# ==============================================================================
# 4. WRITE FAST Q1 TRAINER (train_egy_q1.py)
# ==============================================================================
train_script = r'''#!/usr/bin/env python3
import os, sys, json, time, hashlib, logging
from pathlib import Path
import numpy as np
import torch
import soundfile as sf
import torchaudio

if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]
if not hasattr(torchaudio, "set_audio_backend"):
    torchaudio.set_audio_backend = lambda *args, **kwargs: None

torch.backends.cudnn.benchmark = True
import speechbrain as sb
from hyperpyyaml import load_hyperpyyaml
from speechbrain.utils.distributed import if_main_process

logger = logging.getLogger(__name__)

def _pink_noise(n_samples, gen):
    white = torch.randn(n_samples, generator=gen)
    fft = torch.fft.rfft(white)
    freqs = torch.fft.rfftfreq(n_samples)
    freqs[0] = 1.0
    pink = torch.fft.irfft(fft / torch.sqrt(freqs), n=n_samples)
    return pink / (torch.std(pink) + 1e-8)

def mix_realistic_noise(wav, uid, snr_db, babble_pool=None):
    if snr_db is None:
        return wav
    seed = int(hashlib.md5(f"{uid}_{snr_db}".encode()).hexdigest()[:8], 16)
    gen = torch.Generator().manual_seed(seed)
    n = wav.size(-1)
    pink = _pink_noise(n, gen)
    if babble_pool is not None and len(babble_pool) >= 3:
        idx = torch.randint(0, len(babble_pool), (3,), generator=gen).tolist()
        babble = torch.zeros(n)
        for k in idx:
            src = babble_pool[k]
            if src.numel() == 0:
                continue
            tiled = src.repeat((n // src.numel()) + 2)
            offset = int(torch.randint(0, max(1, tiled.numel() - n), (1,), generator=gen).item())
            babble = babble + tiled[offset:offset + n]
        babble = babble / (torch.std(babble) + 1e-8)
        noise = 0.65 * babble + 0.35 * pink
    else:
        noise = pink
    noise = noise.to(wav.device, dtype=wav.dtype)
    sig_pow = torch.mean(wav ** 2) + 1e-12
    noise_pow = torch.mean(noise ** 2) + 1e-12
    target_noise_pow = sig_pow / (10.0 ** (float(snr_db) / 10.0))
    noisy = wav + torch.sqrt(target_noise_pow / noise_pow) * noise
    peak = torch.max(torch.abs(noisy))
    return noisy * (0.99 / peak) if peak > 0.99 else noisy

def segment_fixed(wav, sr=16000, max_dur=4.0):
    max_s = int(max_dur * sr)
    return [wav[i:i + max_s] for i in range(0, wav.size(-1), max_s)]

def segment_energy(wav, sr=16000, max_dur=4.5, min_dur=2.5, overlap_dur=0.0, hop_size=160, frame_size=400):
    total = wav.size(-1)
    max_s, min_s, ov_s = int(max_dur * sr), int(min_dur * sr), int(overlap_dur * sr)
    if total <= max_s or total < frame_size:
        return [wav]
    unfolded = wav.unfold(-1, frame_size, hop_size)
    energy = torch.sqrt(torch.mean(unfolded ** 2, dim=-1) + 1e-12)
    chunks, start = [], 0
    while start < total:
        if total - start <= max_s:
            chunks.append(wav[start:])
            break
        f_start = (start + min_s) // hop_size
        f_end = min((start + max_s) // hop_size, energy.size(-1))
        split = min(start + max_s, total) if f_end <= f_start else (f_start + torch.argmin(energy[f_start:f_end]).item()) * hop_size
        if split <= start:
            split = min(start + max_s, total)
        chunks.append(wav[start:split])
        start = max(split - ov_s, start + min_s // 2) if ov_s > 0 else split
    return chunks

def merge_overlapping_texts(texts, max_ov_words=5):
    if not texts:
        return ""
    merged = texts[0].split()
    for txt in texts[1:]:
        words = txt.split()
        if not words:
            continue
        best_k = 0
        for k in range(min(max_ov_words, len(merged), len(words)), 0, -1):
            if merged[-k:] == words[:k]:
                best_k = k
                break
        merged.extend(words[best_k:])
    return " ".join(merged)

class ASR(sb.core.Brain):
    def _decode_chunks_batched(self, chunks, current_epoch, searcher, batch_size=32):
        valid_chunks = [c for c in chunks if c.size(0) >= 400]
        if not valid_chunks:
            return []
        texts = []
        for i in range(0, len(valid_chunks), batch_size):
            sub = valid_chunks[i:i + batch_size]
            max_l = max(c.size(0) for c in sub)
            padded = torch.zeros(len(sub), max_l, device=self.device)
            lens = torch.zeros(len(sub), device=self.device)
            for j, c in enumerate(sub):
                padded[j, :c.size(0)] = c
                lens[j] = c.size(0) / max_l
            feats = self.hparams.compute_features(padded)
            feats = self.modules.normalize(feats, lens, epoch=current_epoch)
            src = self.modules.CNN(feats)
            if hasattr(self.modules.Transformer, "encode"):
                enc = self.modules.Transformer.encode(src, wav_len=lens)
            else:
                enc, _ = self.modules.Transformer(src, None, lens)
            hyps, _, _, _ = searcher(enc.detach(), lens)
            for h in hyps:
                t = self.tokenizer.decode_ids(h).strip()
                if t:
                    texts.append(t)
        return texts

    def decode_long_utterance(self, wav, current_epoch, searcher, strategy):
        sr = self.hparams.sample_rate
        if strategy == "fixed":
            chunks = segment_fixed(wav, sr=sr, max_dur=4.0)
            full_text = " ".join(self._decode_chunks_batched(chunks, current_epoch, searcher)).strip()
        elif strategy == "overlap_energy":
            chunks = segment_energy(wav, sr=sr, max_dur=4.5, min_dur=2.5, overlap_dur=0.5)
            full_text = merge_overlapping_texts(self._decode_chunks_batched(chunks, current_epoch, searcher))
        else:
            chunks = segment_energy(wav, sr=sr, max_dur=4.5, min_dur=2.5, overlap_dur=0.0)
            full_text = " ".join(self._decode_chunks_batched(chunks, current_epoch, searcher)).strip()
        return self.tokenizer.encode_as_ids(full_text)

    def compute_forward(self, batch, stage):
        batch = batch.to(self.device)
        wavs, wav_lens = batch.sig
        tokens_bos, _ = batch.tokens_bos

        if stage == sb.Stage.TEST:
            self._test_audio_sec += sum(
                wav_lens[k].item() * wavs.size(1) / self.hparams.sample_rate
                for k in range(wavs.size(0))
            )
            snr_db = getattr(self.hparams, "active_snr_db", None)
            if snr_db is not None:
                babble_pool = getattr(self, "_babble_pool", None)
                wavs = wavs.clone()
                for b in range(wavs.size(0)):
                    b_samp = int(round(wav_lens[b].item() * wavs.size(1)))
                    wavs[b, :b_samp] = mix_realistic_noise(wavs[b, :b_samp], batch.id[b], snr_db, babble_pool)

        strategy = getattr(self.hparams, "active_long_strategy", "standard")
        current_epoch = self.hparams.epoch_counter.current

        if stage == sb.Stage.TEST and strategy in ("fixed", "chunked", "overlap_energy"):
            searcher = self.hparams.test_search
            hyps = []
            for b in range(wavs.size(0)):
                b_samp = int(round(wav_lens[b].item() * wavs.size(1)))
                hyps.append(self.decode_long_utterance(wavs[b, :b_samp], current_epoch, searcher, strategy))
            return None, None, wav_lens, hyps

        try:
            feats = self.hparams.compute_features(wavs)
            feats = self.modules.normalize(feats, wav_lens, epoch=current_epoch)
            if stage == sb.Stage.TRAIN and hasattr(self.hparams, "fea_augment"):
                feats, fea_lens = self.hparams.fea_augment(feats, wav_lens)
                tokens_bos = self.hparams.fea_augment.replicate_labels(tokens_bos)
            src = self.modules.CNN(feats)
            enc_out, pred = self.modules.Transformer(src, tokens_bos, wav_lens, pad_idx=self.hparams.pad_index)
            p_ctc = self.hparams.log_softmax(self.modules.ctc_lin(enc_out))
            p_seq = self.hparams.log_softmax(self.modules.seq_lin(pred))

            hyps = None
            is_valid_search = stage == sb.Stage.VALID and (
                current_epoch % self.hparams.valid_search_interval == 0
                or current_epoch == self.hparams.number_of_epochs
            )
            if is_valid_search:
                hyps, _, _, _ = self.hparams.valid_search(enc_out.detach(), wav_lens)
            elif stage == sb.Stage.TEST:
                hyps, _, _, _ = self.hparams.test_search(enc_out.detach(), wav_lens)
            return p_ctc, p_seq, wav_lens, hyps
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            self._oom_count = getattr(self, "_oom_count", 0) + 1
            return None, None, wav_lens, [[] for _ in range(wavs.size(0))]

    def compute_objectives(self, predictions, batch, stage):
        p_ctc, p_seq, wav_lens, hyps = predictions
        ids = batch.id
        tokens_eos, tokens_eos_lens = batch.tokens_eos
        tokens, tokens_lens = batch.tokens

        if p_ctc is None or p_seq is None:
            if hyps is not None:
                predicted_words = [self.tokenizer.decode_ids(h).split() if h else [] for h in hyps]
                target_words = [w.split() for w in batch.wrd]
                self.wer_metric.append(ids, predicted_words, target_words)
                self.cer_metric.append(ids, predicted_words, target_words)
            return torch.tensor(0.0, device=self.device, requires_grad=True)

        if stage == sb.Stage.TRAIN and hasattr(self.hparams, "fea_augment"):
            rep = self.hparams.fea_augment.replicate_labels
            tokens, tokens_lens = rep(tokens), rep(tokens_lens)
            tokens_eos, tokens_eos_lens = rep(tokens_eos), rep(tokens_eos_lens)
        loss_seq = self.hparams.seq_cost(p_seq, tokens_eos, length=tokens_eos_lens).sum()
        loss_ctc = self.hparams.ctc_cost(p_ctc, tokens, wav_lens, tokens_lens).sum()
        loss = self.hparams.ctc_weight * loss_ctc + (1 - self.hparams.ctc_weight) * loss_seq

        if stage != sb.Stage.TRAIN:
            current_epoch = self.hparams.epoch_counter.current
            do_wer = (
                stage == sb.Stage.TEST
                or current_epoch % self.hparams.valid_search_interval == 0
                or current_epoch == self.hparams.number_of_epochs
            )
            if do_wer and hyps is not None:
                predicted_words = [self.tokenizer.decode_ids(h).split() for h in hyps]
                target_words = [w.split() for w in batch.wrd]
                self.wer_metric.append(ids, predicted_words, target_words)
                self.cer_metric.append(ids, predicted_words, target_words)
            self.acc_metric.append(p_seq, tokens_eos, tokens_eos_lens)
        return loss

    def on_evaluate_start(self, max_key=None, min_key=None):
        super().on_evaluate_start()
        ckpts = self.checkpointer.find_checkpoints(max_key=max_key, min_key=min_key)
        if ckpts:
            ckpt = sb.utils.checkpoints.average_checkpoints(ckpts, recoverable_name="model")
            self.hparams.model.load_state_dict(ckpt, strict=True)
        else:
            self.checkpointer.recover_if_possible()
        self.hparams.model.eval()

    def on_stage_start(self, stage, epoch):
        self._t0 = time.time()
        self._test_audio_sec = 0.0
        self._oom_count = 0
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        if stage != sb.Stage.TRAIN:
            self.acc_metric = self.hparams.acc_computer()
            self.wer_metric = self.hparams.error_rate_computer()
            self.cer_metric = self.hparams.error_rate_computer(split_tokens=True)

    def _log_efficiency(self, stage, epoch):
        if not if_main_process():
            return
        elapsed = round(time.time() - self._t0, 2)
        rtf = round(elapsed / self._test_audio_sec, 4) if (stage == sb.Stage.TEST and self._test_audio_sec > 0) else None
        rec = {
            "stage": str(stage).split(".")[-1], "epoch": epoch, "seconds": elapsed,
            "audio_sec": round(self._test_audio_sec, 1) if stage == sb.Stage.TEST else None,
            "rtf": rtf,
            "peak_gpu_mb": round(torch.cuda.max_memory_allocated() / 2**20) if torch.cuda.is_available() else None,
            "oom_count": getattr(self, "_oom_count", 0),
            "test_name": getattr(self.hparams, "test_name", None) if stage == sb.Stage.TEST else None,
        }
        with open(os.path.join(self.hparams.output_folder, "efficiency.jsonl"), "a") as f:
            f.write(json.dumps(rec) + "\n")

    def on_stage_end(self, stage, stage_loss, epoch):
        self._log_efficiency(stage, epoch)
        stage_stats = {"loss": stage_loss}
        if stage == sb.Stage.TRAIN:
            self.train_stats = stage_stats
            return
        if self.acc_metric.correct + self.acc_metric.total > 0:
            stage_stats["ACC"] = self.acc_metric.summarize()
        current_epoch = self.hparams.epoch_counter.current
        do_wer = (
            stage == sb.Stage.TEST
            or current_epoch % self.hparams.valid_search_interval == 0
            or current_epoch == self.hparams.number_of_epochs
        )
        if do_wer:
            stage_stats["WER"] = self.wer_metric.summarize("error_rate")
            stage_stats["CER"] = self.cer_metric.summarize("error_rate")
        if stage == sb.Stage.VALID:
            lr = self.hparams.noam_annealing.current_lr
            self.hparams.train_logger.log_stats(
                stats_meta={"epoch": epoch, "lr": lr, "steps": self.optimizer_step},
                train_stats=self.train_stats, valid_stats=stage_stats,
            )
            self.checkpointer.save_and_keep_only(
                meta={"ACC": stage_stats.get("ACC", 0.0), "epoch": epoch},
                max_keys=["ACC"], num_to_keep=self.hparams.avg_checkpoints,
            )
        elif stage == sb.Stage.TEST:
            self.hparams.train_logger.log_stats(
                stats_meta={"Epoch": self.hparams.epoch_counter.current, "test": self.hparams.test_name},
                test_stats=stage_stats,
            )
            if if_main_process():
                with open(self.hparams.test_wer_file, "w", encoding="utf-8") as w:
                    self.wer_metric.write_stats(w)
                cer_path = os.path.join(
                    os.path.dirname(self.hparams.test_wer_file),
                    os.path.basename(self.hparams.test_wer_file).replace("wer_", "cer_")
                )
                with open(cer_path, "w", encoding="utf-8") as w:
                    self.cer_metric.write_stats(w)

    def on_fit_batch_end(self, batch, outputs, loss, should_step):
        if should_step:
            self.hparams.noam_annealing(self.optimizer)

def dataio_prepare(hparams, tokenizer):
    data_folder = hparams["data_folder"]
    load = lambda p: sb.dataio.dataset.DynamicItemDataset.from_csv(csv_path=p, replacements={"data_root": data_folder})
    train_data = load(hparams["train_csv"])
    valid_data = load(hparams["valid_csv"]).filtered_sorted(sort_key="duration")
    test_datasets = {Path(p).stem: load(p).filtered_sorted(sort_key="duration") for p in hparams["test_csv"]}
    datasets = [train_data, valid_data] + list(test_datasets.values())
    valtest = [valid_data] + list(test_datasets.values())

    def read(wav):
        x, sr = sf.read(wav, dtype="float32", always_2d=False)
        return torch.from_numpy(x)

    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def audio_pipeline(wav):
        return read(wav)
    sb.dataio.dataset.add_dynamic_item(valtest, audio_pipeline)

    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def audio_pipeline_train(wav):
        sig = read(wav)
        if "speed_perturb" in hparams:
            sig = hparams["speed_perturb"](sig.unsqueeze(0)).squeeze(0)
        return sig
    sb.dataio.dataset.add_dynamic_item([train_data], audio_pipeline_train)

    @sb.utils.data_pipeline.takes("wrd")
    @sb.utils.data_pipeline.provides("wrd", "tokens_list", "tokens_bos", "tokens_eos", "tokens")
    def text_pipeline(wrd):
        yield wrd
        tokens_list = tokenizer.encode_as_ids(wrd)
        yield tokens_list
        yield torch.LongTensor([hparams["bos_index"]] + tokens_list)
        yield torch.LongTensor(tokens_list + [hparams["eos_index"]])
        yield torch.LongTensor(tokens_list)
    sb.dataio.dataset.add_dynamic_item(datasets, text_pipeline)
    sb.dataio.dataset.set_output_keys(datasets, ["id", "sig", "wrd", "tokens_bos", "tokens_eos", "tokens"])
    return train_data, valid_data, test_datasets

def build_babble_pool(train_csv, data_folder, max_clips=40):
    import pandas as pd
    df = pd.read_csv(train_csv)
    df = df[df["duration"].astype(float) >= 2.0].head(max_clips)
    pool = []
    for p in df["wav"].tolist():
        wav_path = p.replace("$data_root", data_folder)
        if os.path.exists(wav_path):
            x, _ = sf.read(wav_path, dtype="float32", always_2d=False)
            pool.append(torch.from_numpy(x))
    return pool

if __name__ == "__main__":
    hparams_file, run_opts, overrides = sb.parse_arguments(sys.argv[1:])
    with open(hparams_file, encoding="utf-8") as fin:
        hparams = load_hyperpyyaml(fin, overrides)
    sb.utils.distributed.ddp_init_group(run_opts)
    sb.create_experiment_directory(
        experiment_directory=hparams["output_folder"],
        hyperparams_to_save=hparams_file, overrides=overrides,
    )
    tokenizer = hparams["tokenizer"]
    tokenizer.load(hparams["bpe_model"])
    train_data, valid_data, test_datasets = dataio_prepare(hparams, tokenizer)
    hparams["test_search"] = hparams["valid_search"]
    hparams["test_search"].beam_size = hparams["test_beam_size"]

    asr_brain = ASR(
        modules=hparams["modules"], opt_class=hparams["Adam"],
        hparams=hparams, run_opts=run_opts, checkpointer=hparams["checkpointer"],
    )
    asr_brain.tokenizer = tokenizer

    n_params = sum(p.numel() for p in asr_brain.hparams.model.parameters())
    Path(hparams["output_folder"], "params.json").write_text(json.dumps({"params": n_params}))

    if not hparams["skip_train"]:
        asr_brain.fit(
            asr_brain.hparams.epoch_counter, train_data, valid_data,
            train_loader_kwargs=hparams["train_dataloader_opts"],
            valid_loader_kwargs=hparams["valid_dataloader_opts"],
        )

    os.makedirs(hparams["output_wer_folder"], exist_ok=True)
    strategies = [s.strip() for s in str(hparams.get("long_strategies", "chunked")).split(",") if s.strip()]
    snr_raw = str(hparams.get("snr_levels", "")).strip()
    snr_list = [int(x) for x in snr_raw.split(",") if x.strip()] if snr_raw else []

    for name, data in test_datasets.items():
        asr_brain.hparams.active_snr_db = None
        if "long" in name:
            for strat in strategies:
                asr_brain.hparams.active_long_strategy = strat
                asr_brain.hparams.test_name = f"{name}_{strat}"
                asr_brain.hparams.test_wer_file = os.path.join(hparams["output_wer_folder"], f"wer_{name}_{strat}.txt")
                test_opts = dict(hparams["test_dataloader_opts"])
                test_opts["batch_size"] = 1
                asr_brain.evaluate(data, max_key="ACC", test_loader_kwargs=test_opts)
        else:
            asr_brain.hparams.active_long_strategy = "standard"
            asr_brain.hparams.test_name = name
            asr_brain.hparams.test_wer_file = os.path.join(hparams["output_wer_folder"], f"wer_{name}.txt")
            asr_brain.evaluate(data, max_key="ACC", test_loader_kwargs=dict(hparams["test_dataloader_opts"]))

    if snr_list and "test" in test_datasets:
        asr_brain._babble_pool = build_babble_pool(hparams["train_csv"], hparams["data_folder"])
        asr_brain.hparams.active_long_strategy = "standard"
        for snr in snr_list:
            asr_brain.hparams.active_snr_db = snr
            asr_brain.hparams.test_name = f"test_snr{snr}"
            asr_brain.hparams.test_wer_file = os.path.join(hparams["output_wer_folder"], f"wer_test_snr{snr}.txt")
            asr_brain.evaluate(test_datasets["test"], max_key="ACC", test_loader_kwargs=dict(hparams["test_dataloader_opts"]))
'''
Path(f"{REPO}/train_egy_q1.py").write_text(train_script)

# ==============================================================================
# 5. LOCATE DATASET & AUTO-DETECT ANY SAVED CHECKPOINTS
# ==============================================================================
bpe_candidates = sorted(glob.glob("/kaggle/input/**/tokenizer/bpe500.model", recursive=True))
assert bpe_candidates, "ejust-prepared-v1 not found! Add Input -> Datasets -> ejust-prepared-v1."
BPE = bpe_candidates[0]
DATA = os.path.dirname(os.path.dirname(BPE))
print(f"Dataset root: {DATA}")

def find_saved_ckpt(run_id):
    roots = [EXISTING_CKPTS_DIR] if EXISTING_CKPTS_DIR else ["/kaggle/input", "/kaggle/working"]
    for root in roots:
        if not root or not os.path.isdir(root):
            continue
        for hit in sorted(glob.glob(f"{root}/**/{run_id}", recursive=True)):
            if glob.glob(f"{hit}/save/CKPT*"):
                return hit
    return None

GLOBAL_T0 = time.time()
TIME_LIMIT_SEC = int(TIME_LIMIT_H * 3600)

# ==============================================================================
# 6. ROCK-SOLID PURE PYTORCH WHISPER-SMALL LoRA WORKER (~18 MIN ON 1x T4)
# ==============================================================================
whisper_worker_code = r'''
import os, re, sys, json, time, math
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from torch.utils.data import Dataset, DataLoader
from jiwer import wer as jiwer_wer, cer as jiwer_cer
from transformers import WhisperProcessor, WhisperForConditionalGeneration, get_cosine_schedule_with_warmup

# CRITICAL FIX: Bypass broken torchao check in peft
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
from peft import LoraConfig, get_peft_model

DATA, MODEL_ID, OUT_DIR, EPOCHS, LR, BS = sys.argv[1], sys.argv[2], sys.argv[3], int(sys.argv[4]), float(sys.argv[5]), int(sys.argv[6])
os.makedirs(OUT_DIR, exist_ok=True)
device = torch.device("cuda:0")

def norm_ar(s):
    s = re.sub(r"[\u0617-\u061A\u064B-\u0652\u0640]", "", str(s))
    s = re.sub(r"[^\w\s\u0600-\u06FF]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="Arabic", task="transcribe")
fe = processor.feature_extractor
tok = processor.tokenizer
pad_id = tok.pad_token_id

class WhisperDS(Dataset):
    def __init__(self, csv_path, data_root):
        df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
        self.wavs = [p.replace("$data_root", data_root) for p in df["wav"].tolist()]
        self.texts = df["wrd"].tolist()
    def __len__(self):
        return len(self.wavs)
    def __getitem__(self, idx):
        x, _ = sf.read(self.wavs[idx], dtype="float32", always_2d=False)
        mel = fe(x, sampling_rate=16000).input_features[0]
        ids = tok(self.texts[idx]).input_ids
        return mel, ids

def collate_fn(batch):
    mels = torch.tensor(np.stack([b[0] for b in batch]), dtype=torch.float32)
    max_len = max(len(b[1]) for b in batch)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, (_, ids) in enumerate(batch):
        labels[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
    if (labels[:, 0] == tok.bos_token_id).all():
        labels = labels[:, 1:]
    return mels, labels

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.use_cache = False
lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
model = get_peft_model(model, lora_cfg).to(device)

total_p = sum(p.numel() for p in model.parameters()) / 1e6
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"[Whisper-LoRA] Loaded {MODEL_ID} | Total={total_p:.1f}M | Trainable LoRA={train_p:.2f}M", flush=True)

train_ds = WhisperDS(f"{DATA}/train.csv", DATA)
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=4, pin_memory=True, collate_fn=collate_fn)

grad_accum = max(1, 32 // BS)
total_steps = math.ceil(len(train_loader) / grad_accum) * EPOCHS
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda")

t_train = time.time()
model.train()
for ep in range(1, EPOCHS + 1):
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    for step, (mels, labels) in enumerate(train_loader, 1):
        mels = mels.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            out = model(input_features=mels, labels=labels)
            loss = out.loss / grad_accum
        scaler.scale(loss).backward()
        running_loss += loss.item() * grad_accum
        if step % grad_accum == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        if step % 400 == 0 or step == len(train_loader):
            print(f"[Whisper-LoRA] Epoch {ep}/{EPOCHS} | step {step}/{len(train_loader)} | loss={running_loss/step:.4f}", flush=True)

train_min = round((time.time() - t_train) / 60.0, 1)
print(f"[Whisper-LoRA] Training finished in {train_min} min. Merging LoRA weights...", flush=True)

merged = model.merge_and_unload().eval().half().to(device)
forced_ids = processor.get_decoder_prompt_ids(language="arabic", task="transcribe")

@torch.no_grad()
def eval_split(split_name, eval_bs=32):
    df = pd.read_csv(f"{DATA}/{split_name}.csv", dtype=str, keep_default_na=False)
    wavs = [p.replace("$data_root", DATA) for p in df["wav"].tolist()]
    refs = [norm_ar(r) for r in df["wrd"].tolist()]
    total_audio = df["duration"].astype(float).sum()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    hyps = []

    if "long" not in split_name:
        for i in range(0, len(wavs), eval_bs):
            sub_wavs = [sf.read(p, dtype="float32", always_2d=False)[0] for p in wavs[i:i+eval_bs]]
            mels = fe(sub_wavs, sampling_rate=16000, return_tensors="pt").input_features.half().to(device)
            gen_ids = merged.generate(mels, forced_decoder_ids=forced_ids, max_new_tokens=128)
            hyps.extend([norm_ar(t) for t in tok.batch_decode(gen_ids, skip_special_tokens=True)])
    else:
        chunk_s = 28 * 16000
        for p in wavs:
            x, _ = sf.read(p, dtype="float32", always_2d=False)
            chunks = [x[k:k+chunk_s] for k in range(0, len(x), chunk_s) if len(x[k:k+chunk_s]) >= 1600]
            if not chunks:
                chunks = [x]
            mels = fe(chunks, sampling_rate=16000, return_tensors="pt").input_features.half().to(device)
            gen_ids = merged.generate(mels, forced_decoder_ids=forced_ids, max_new_tokens=220)
            txt = " ".join(tok.batch_decode(gen_ids, skip_special_tokens=True))
            hyps.append(norm_ar(txt))

    elapsed = time.time() - t0
    valid_pairs = [(r, h) for r, h in zip(refs, hyps) if r.strip()]
    if valid_pairs:
        v_refs, v_hyps = zip(*valid_pairs)
        w = round(jiwer_wer(list(v_refs), list(v_hyps)) * 100, 2)
        c = round(jiwer_cer(list(v_refs), list(v_hyps)) * 100, 2)
    else:
        w, c = 0.0, 0.0
    rtf = round(elapsed / total_audio, 4)
    vram = round(torch.cuda.max_memory_allocated() / 2**20)
    print(f"[Whisper-LoRA] {split_name:10s} | WER={w:.2f}% | CER={c:.2f}% | RTF={rtf:.4f} | PeakVRAM={vram}MB", flush=True)
    return w, c, rtf, vram

t_wer, t_cer, t_rtf, t_vram = eval_split("test", eval_bs=32)
l_wer, l_cer, l_rtf, l_vram = eval_split("test_long")

res = {
    "model": "Whisper-Small + LoRA (FT)", "params_m": round(total_p, 1),
    "trainable_m": round(train_p, 2), "train_min": train_min,
    "test_wer": t_wer, "test_cer": t_cer, "rtf_test": t_rtf, "vram_test_mb": t_vram,
    "test_long_wer": l_wer, "test_long_cer": l_cer, "rtf_long": l_rtf, "vram_long_mb": l_vram,
}
Path(f"{OUT_DIR}/whisper_lora_results.json").write_text(json.dumps(res, indent=2))
'''
Path(f"{WORK}/run_whisper_lora.py").write_text(whisper_worker_code)

# ==============================================================================
# 7. NON-COLLIDING DUAL-GPU SCHEDULER (GPU QUEUE)
# ==============================================================================
gpu_queue = queue.Queue()
for g in range(NUM_GPUS):
    gpu_queue.put(g)

def run_subprocess_verbose(cmd, env, cwd, log_path, tag):
    t0 = time.time()
    tail_lines = []
    with open(log_path, "w") as logf:
        proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            logf.write(line)
            tail_lines.append(line)
            if len(tail_lines) > 35:
                tail_lines.pop(0)
            if any(k in line for k in ("train_logger", "[Whisper-LoRA]", "ABLATION", "ERROR", "Exception")):
                print(f"[{tag}] {line}", end="", flush=True)
            if time.time() - GLOBAL_T0 > TIME_LIMIT_SEC:
                print(f"[{tag}] TIME LIMIT REACHED — stopping cleanly.", flush=True)
                proc.terminate()
                break
        proc.wait()
    mins = (time.time() - t0) / 60.0
    if proc.returncode != 0:
        print(f"\n[FAILED {tag} | exit={proc.returncode} | {mins:.1f} min] Last log lines:\n" + "".join(tail_lines), flush=True)
    else:
        print(f"[DONE {tag} | exit=0 | {mins:.1f} min]", flush=True)
    return proc.returncode

def execute_job(job):
    gpu_id = gpu_queue.get()
    try:
        kind, run_id, cfg = job
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        if kind == "whisper":
            out_json = f"{RESULTS_ROOT}/whisper_lora_results.json"
            if os.path.exists(out_json):
                print(f"[GPU{gpu_id}] SKIP Whisper-LoRA (already exists).")
                return run_id, 0
            cmd = [sys.executable, f"{WORK}/run_whisper_lora.py", DATA, WHISPER_MODEL, RESULTS_ROOT, str(WHISPER_EPOCHS), str(WHISPER_LR), str(WHISPER_BS)]
            print(f"[GPU{gpu_id}] START Whisper-Small LoRA ({WHISPER_EPOCHS} ep)...", flush=True)
            rc = run_subprocess_verbose(cmd, env, WORK, f"{WORK}/whisper_lora.log", f"Whisper|GPU{gpu_id}")
            return run_id, rc
        else:
            out_dir = f"{RESULTS_ROOT}/{run_id}"
            if cfg.get("restore_from"):
                shutil.copytree(cfg["restore_from"], out_dir, dirs_exist_ok=True)
            hp_path = f"{WORK}/{run_id}.yaml"
            ejust_tools.make_hparams(
                f"{REPO}/hparams/S2S/{cfg['template']}.yaml", hp_path,
                data_folder=DATA, csv_dir=DATA, bpe_model=BPE, output_folder=out_dir,
                epochs=cfg["epochs"], warmup=WARMUP_STEPS,
                valid_search_interval=VALID_SEARCH_INTERVAL, test_beam_size=TEST_BEAM_SIZE,
                batch_size=TRAIN_BATCH_SIZE, lr=TRAIN_LR, num_workers=4,
                long_strategies=",".join(cfg["long_strategies"]),
                snr_levels=",".join(str(x) for x in cfg["snr_levels"]),
                skip_train=cfg["skip_train"],
                ablation_overrides=cfg.get("overrides"),
            )
            cmd = [sys.executable, "train_egy_q1.py", hp_path, "--precision", "fp16", "--seed", str(cfg["seed"])]
            print(f"[GPU{gpu_id}] START {run_id} (template={cfg['template']}, BS={TRAIN_BATCH_SIZE}, skip_train={cfg['skip_train']})", flush=True)
            rc = run_subprocess_verbose(cmd, env, REPO, f"{WORK}/{run_id}.log", f"{run_id}|GPU{gpu_id}")
            return run_id, rc
    finally:
        gpu_queue.put(gpu_id)

# --- Build Job List ---
jobs = []

if RUN_WHISPER_LORA:
    jobs.append(("whisper", "whisper_lora", {}))

new_ablation_specs = []
if RUN_NEW_ABLATIONS:
    new_ablation_specs.append(("conmamba_k7_s7775", "conmamba_small", 7775, {"kernel_size": 7}, "w/ Narrow Conv Kernel (K=7 vs K=31)"))
    new_ablation_specs.append(("bimamba_noconv_s7775", "conmamba_small", 7775, {"encoder_module": "mamba"}, "w/o Depthwise Conv (Pure BiMamba)"))

    for rid, tmpl, seed, ov, _ in new_ablation_specs:
        jobs.append(("sb", rid, dict(
            template=tmpl, seed=seed, epochs=ABLATION_EPOCHS,
            long_strategies=["chunked"], snr_levels=[], skip_train=False, overrides=ov,
        )))

if RUN_CKPT_INFERENCE:
    for base_m in ["conmamba_small", "conformer_small"]:
        rid = f"{base_m}_s7775"
        ckpt_dir = find_saved_ckpt(rid)
        if ckpt_dir:
            print(f"[AUTO-FOUND CKPT] {rid} -> {ckpt_dir}")
            jobs.append(("sb", rid, dict(
                template=base_m, seed=7775, epochs=110, restore_from=ckpt_dir,
                long_strategies=LONG_STRATEGIES, snr_levels=SNR_LEVELS_DB, skip_train=True, overrides=None,
            )))
        else:
            print(f"[INFO] No saved checkpoint found for {rid} (Table IV / Noise inference skipped).")

print(f"\n=== Running {len(jobs)} Q1 Jobs safely across {NUM_GPUS} GPU(s) (No Collisions) ===")
with ThreadPoolExecutor(max_workers=NUM_GPUS) as ex:
    futs = [ex.submit(execute_job, j) for j in jobs]
    for fut in as_completed(futs):
        fut.result()

# ==============================================================================
# 8. ENCODER LATENCY & PEAK VRAM SCALING BENCHMARK (5s -> 60s ON T4)
# ==============================================================================
latency_rows = []
if RUN_LATENCY_BENCHMARK:
    print("\n=== Running Encoder Latency & Peak VRAM Scaling Benchmark (5s - 60s) ===", flush=True)
    from hyperpyyaml import load_hyperpyyaml
    for m_name in ["conmamba_small", "conformer_small"]:
        hp_tmp = f"{WORK}/bench_{m_name}.yaml"
        ejust_tools.make_hparams(
            f"{REPO}/hparams/S2S/{m_name}.yaml", hp_tmp,
            data_folder=DATA, csv_dir=DATA, bpe_model=BPE, output_folder=f"{WORK}/tmp_bench",
            epochs=1, warmup=100, valid_search_interval=1, skip_train=True,
        )
        with open(hp_tmp) as f:
            hp = load_hyperpyyaml(f)
        feat_fn = hp["compute_features"].cuda().eval()
        cnn = hp["modules"]["CNN"].cuda().eval()
        trans = hp["modules"]["Transformer"].cuda().eval()

        for dur in [5, 15, 30, 45, 60]:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            wav = torch.randn(1, int(dur * 16000), device="cuda")
            lens = torch.tensor([1.0], device="cuda")
            try:
                with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
                    feats = feat_fn(wav)
                    for _ in range(3):
                        src = cnn(feats)
                        _ = trans.encode(src, wav_len=lens) if hasattr(trans, "encode") else trans(src, None, lens)
                    torch.cuda.synchronize()
                    times = []
                    for _ in range(8):
                        t0 = time.perf_counter()
                        src = cnn(feats)
                        _ = trans.encode(src, wav_len=lens) if hasattr(trans, "encode") else trans(src, None, lens)
                        torch.cuda.synchronize()
                        times.append((time.perf_counter() - t0) * 1000.0)
                latency_rows.append({
                    "model": m_name, "duration_s": dur,
                    "latency_ms": round(float(np.median(times)), 1),
                    "peak_vram_mb": round(torch.cuda.max_memory_allocated() / 2**20, 1),
                })
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                latency_rows.append({"model": m_name, "duration_s": dur, "latency_ms": "OOM", "peak_vram_mb": "OOM"})
        del feat_fn, cnn, trans; torch.cuda.empty_cache()

# ==============================================================================
# 9. CONSOLIDATED Q1 TABLES & IEEE TASLP LATEX OUTPUT
# ==============================================================================
def parse_wer(path):
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as fin:
        first_line = fin.readline()
    m = re.search(r"%WER\s+([\d\.]+)", first_line)
    return float(m.group(1)) if m else None

# --- TABLE A: Main + Whisper LoRA ---
w_path = f"{RESULTS_ROOT}/whisper_lora_results.json"
w_res = json.loads(Path(w_path).read_text()) if os.path.exists(w_path) else None
main_rows = [COMPLETED_MAIN[0]] + ([w_res] if w_res else []) + COMPLETED_MAIN[1:]
df_main = pd.DataFrame(main_rows)
print("\n" + "=" * 95)
print("Q1 TABLE A: MAIN BENCHMARK (INCLUDING FINE-TUNED WHISPER-SMALL + LoRA)")
print("=" * 95)
print(df_main[["model", "params_m", "test_wer", "test_cer", "test_long_wer", "test_long_cer", "rtf_long"]].to_string(index=False))

# --- TABLE B: Complete Architectural Ablation Study ---
abl_rows = list(COMPLETED_ABLATIONS)
for rid, _, _, _, label in new_ablation_specs:
    od = f"{RESULTS_ROOT}/{rid}"
    wd = f"{od}/wer_output" if os.path.isdir(f"{od}/wer_output") else od
    if os.path.isdir(od):
        pm = round(json.loads(Path(od, "params.json").read_text())["params"] / 1e6, 1) if Path(od, "params.json").exists() else "-"
        abl_rows.append({
            "Variant": label, "Params (M)": pm,
            "Test WER (%)": parse_wer(f"{wd}/wer_test.txt"),
            "Test CER (%)": parse_wer(f"{wd}/cer_test.txt"),
            "Long WER (%)": parse_wer(f"{wd}/wer_test_long_chunked.txt"),
        })
df_abl = pd.DataFrame(abl_rows)
print("\n" + "=" * 95)
print("Q1 TABLE B: ARCHITECTURAL ABLATION STUDY (PROVES CONTRIBUTION #3)")
print("=" * 95)
print(df_abl.to_string(index=False))

# --- TABLE C: Latency & Peak VRAM Scaling ---
if latency_rows:
    df_lat = pd.DataFrame(latency_rows)
    print("\n" + "=" * 95)
    print("Q1 TABLE C: ENCODER LATENCY (ms) & PEAK VRAM (MB) SCALING ON TESLA T4")
    print("=" * 95)
    print(df_lat.pivot(index="duration_s", columns="model", values=["latency_ms", "peak_vram_mb"]).to_string())

# --- Export IEEE Booktabs LaTeX ---
tex = []
tex.append(r"% === TABLE: MAIN BENCHMARK + FINE-TUNED WHISPER-LoRA ===")
tex.append(r"\begin{table}[t]\centering")
tex.append(r"\caption{Benchmark comparison on curated EJUST Egyptian Arabic (\texttt{test} and \texttt{test\_long}).}")
tex.append(r"\begin{tabular}{lcccccc}\toprule")
tex.append(r"\textbf{Architecture} & \textbf{Params} & \textbf{Test WER} & \textbf{Test CER} & \textbf{Long WER} & \textbf{Long CER} & \textbf{RTF} \\ \midrule")
for _, r in df_main.iterrows():
    m = str(r["model"]).replace("_", "\\_")
    tex.append(f"{m} & {r['params_m']}M & {r['test_wer']} & {r['test_cer']} & {r['test_long_wer']} & {r['test_long_cer']} & {r['rtf_long']} \\\\")
tex.append(r"\bottomrule\end{tabular}\end{table}")

tex.append(r"\n% === TABLE: ARCHITECTURAL ABLATION ===")
tex.append(r"\begin{table}[t]\centering")
tex.append(r"\caption{Architectural ablation of ConMamba-Small on EJUST Egyptian Arabic.}")
tex.append(r"\begin{tabular}{lcccc}\toprule")
tex.append(r"\textbf{Configuration} & \textbf{Params} & \textbf{Test WER (\%)} & \textbf{Test CER (\%)} & \textbf{Long WER (\%)} \\ \midrule")
for _, r in df_abl.iterrows():
    v = str(r["Variant"]).replace("_", "\\_")
    tex.append(f"{v} & {r['Params (M)']}M & {r['Test WER (%)']} & {r['Test CER (%)']} & {r['Long WER (%)']} \\\\")
tex.append(r"\bottomrule\end{tabular}\end{table}")

tex_str = "\n".join(tex)
Path(f"{RESULTS_ROOT}/q1_tables_v2.tex").write_text(tex_str)
print("\n" + tex_str)
print(f"\nAll Q1 outputs saved to: {RESULTS_ROOT}")

Python 3.13.16 | PyTorch 2.11.0+cu130 | GPUs: 2x Tesla T4
Available Mamba-ASR S2S templates: ['conformer_large', 'conformer_small', 'conmamba_large', 'conmamba_small', 'conmambamamba_large', 'conmambamamba_small']
Patched Mamba-ASR successfully.
Dataset root: /kaggle/input/notebooks/monaygad/egyptian-arabic-asr-masry-prepared/ejust_prepared_v1
[INFO] No saved checkpoint found for conmamba_small_s7775 (Table IV / Noise inference skipped).
[INFO] No saved checkpoint found for conformer_small_s7775 (Table IV / Noise inference skipped).

=== Running 3 Q1 Jobs safely across 2 GPU(s) (No Collisions) ===
[GPU0] START Whisper-Small LoRA (3 ep)...
[ABLATION] conmamba_k7_s7775.yaml: injected kernel_size: 7 into Transformer block
[GPU1] START conmamba_k7_s7775 (template=conmamba_small, BS=64, skip_train=False)
[Whisper|GPU0] [Whisper-LoRA] Loaded openai/whisper-small | Total=243.5M | Trainable LoRA=1.77M
[conmamba_k7_s7775|GPU1] speechbrain.utils.train_logger - epoch: 1, lr: 4.09e-05, steps: 410 

/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:186: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:255: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:323: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:396: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/kaggle/working/Mamba-ASR/modules/mamba/selective_scan_interface.py:468: FutureWarning: `torch.c

dropout=0.1 is not used in Mamba.


/usr/local/lib/python3.13/dist-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)



Q1 TABLE A: MAIN BENCHMARK (INCLUDING FINE-TUNED WHISPER-SMALL + LoRA)
                    model  params_m  test_wer  test_cer  test_long_wer  test_long_cer  rtf_long
Whisper-Small (Zero-Shot)     241.7     71.63     33.64          53.62          21.16    0.0966
Whisper-Small + LoRA (FT)     243.5     15.53      4.45          18.54           5.77    0.0931
      ConMambaMamba-Small      12.2     17.97      6.41          19.82           7.44    0.0512
          Conformer-Small      12.5     14.49      4.77          15.38           4.82    0.0624
    ConMamba-Small (Ours)      12.1     13.85      4.44          14.65           4.57    0.0438

Q1 TABLE B: ARCHITECTURAL ABLATION STUDY (PROVES CONTRIBUTION #3)
                                         Variant  Params (M)  Test WER (%)  Test CER (%)  Long WER (%)
ConMamba-Small (Full: Bi-SSM + K=31 + CTC a=0.3)        12.1         13.85          4.44         14.65
          w/o Bidirectional (Unidirectional SSM)        10.6         14.97     